# Knee v6: K=2 screen on fold 0 (two series per plane), ConvNeXt-T
Needs the committed output of `knee-v6-cache-k2` (cache with 6 series slots per study) attached next to the usual inputs; the notebook finds it by looking for a `cache_index.csv` that has a `present_5` column.
Two arms, both `--K 2`, `convnext_tiny.in12k_ft_in1k`, lr_backbone 5e-5, fold 0, 12 epochs:
  GPU 0: all 16 slices per series (96 windows per study, twice the K=1 work) with `--bs 2 --accum 4`  ~900 s/epoch, ~3 h
  GPU 1: 8 slices per series (48 windows per study, same work per step as K=1) with `--bs 4 --accum 2`  ~450 s/epoch, ~1.5 h
Both keep ~192 windows per micro-batch, the size that already fits a T4 for ConvNeXt-T.
Reference (K=1, same recipe): fold-0 macro 0.833 / KEY-4 0.835 (rank table); raw epoch curve ep1 0.700 | ep3 0.763 | ep5 0.800 | ep9 0.828 | ep12 0.832. Noise ~0.01: act only on a gain above ~+0.015.
Watchdog (v6_6): `train.py` now prints a heartbeat every 25 micro-batches (data-wait vs compute seconds, projected epoch time). The watchdog counts heartbeats as progress: no first heartbeat in 15 min, or a 15 min stall, kills the arm; an arm whose first epoch exceeds 1,300 s, or whose projected epoch time (read from the heartbeat after 50 micro-batches) exceeds 2,600 s, is killed. A probe cell (cell 3) times disk read, CPU augment, the DataLoader and the GPU step for both arms before training starts; read its output if an arm is slow.
The previous run of this notebook stalled with no epoch in 45 min on both arms; cause not confirmed (suspects: reading the 16 GB K=2 cache from /kaggle/input, and 6 processes on 4 vCPUs). The probe and heartbeats are there to find out.
Inputs:The competition data, knee-mri-pipeline, with the latest zip (knee_pipeline_v6_6: v6_5 plus a heartbeat and `--torch_threads` in train.py, and probe_loader.py). ConvNeXt only needs the v6 code, but use one version consistently.
The committed output of knee-v6-cache-k2, which holds the K=2 cache.The output of knee-mri-diagnosis-caching, for labels_soft.csv. This also mounts the old K=1 cache, which is fine. The screen finds the K=2 cache by looking for a cache_index.csv with a present_5 column, and the K=1 index has no such column.The output of knee-mri-diagnosis-training, for oof_all.csv (the baseline).
The output of the ConvNeXt 5-fold notebook (knee-v6-convnext-5fold), for oof_v6convnext.csv. It’s used only in the final comparison table. Without it the table is skipped, and the arms still train.
GPU T4 x2, internet ON, Save & Run All. ~3 h wall.

In [1]:
import os, sys, re, json, subprocess, time, threading
from pathlib import Path
import numpy as np, pandas as pd

CNX_T = "convnext_tiny.in12k_ft_in1k"
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE_ROOT = Path("/kaggle/input/notebooks")   # notebook outputs only: walking the whole of /kaggle/input (DICOM folders) takes minutes
_c = [p.parent for p in CACHE_ROOT.rglob("cache_index.csv") if "present_5" in pd.read_csv(p, nrows=1).columns]
assert len(_c) == 1, f"need exactly one K=2 cache (cache_index.csv with present_5), found {_c}"
CACHE = _c[0]
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "tests/smoke_v6.py", "probe_loader.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6_6 (attach the latest dataset version)"
print("K=2 cache:", CACHE, "| backbone", CNX_T)

K=2 cache: /kaggle/input/notebooks/theokorir/knee-v6-k2-cache/cache | backbone convnext_tiny.in12k_ft_in1k


In [2]:
# tests: if any fails, the assert stops the notebook BEFORE training starts
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens"]
print("extra train args:", EXTRA)

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens']


In [3]:
# probe: time disk read / CPU augment / DataLoader / GPU step for both arms at once (they share the machine in the real run).
# Informational only: it never stops the notebook. If stage 1 or 3 is the slowest, the cache read or the CPUs are the bottleneck.
PROBE = {0: ["--bs", "2", "--train_slices", "16"], 1: ["--bs", "4", "--train_slices", "8"]}
procs = {}
for gpu, extra in PROBE.items():
    cmd = [sys.executable, str(CODE / "probe_loader.py"), "--cache_dir", str(CACHE), "--labels", str(LABELS), "--K", "2",
           "--backbone", CNX_T, "--head", "tokens", "--workers", "2", "--n", "24"] + extra
    procs[gpu] = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(CODE),
                                  env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="1"))
for gpu, p in procs.items():
    try:
        out, _ = p.communicate(timeout=900)
    except subprocess.TimeoutExpired:
        p.kill(); out, _ = p.communicate(); out += "\n[probe killed after 900 s]"
    print(f"== probe, GPU {gpu}, args {PROBE[gpu]} (exit {p.returncode})")
    print("\n".join(l for l in out.splitlines() if not l.startswith(("Warning", "/kaggle", "  self."))))


== probe, GPU 0, args ['--bs', '2', '--train_slices', '16'] (exit 0)
cache /kaggle/input/notebooks/theokorir/knee-v6-k2-cache/cache | 4407 studies | K=2 bs=2 ts=16 workers=2 | 1 torch threads

1 read    : 0.058 s/study | 63 MB/s | 3.7 MB/study | one worker alone would need 3 min/epoch
2 cpu     : 0.180 s/study (convert + augment, 1 process) | 2 workers: 5 min/epoch if nothing else competes
3 loader  : first batch 0.6s | 0.32 s/batch | projected 9 min/epoch
4 gpu     : 0.39 s/batch | projected 12 min/epoch | peak mem 6.7 GB

Epoch time is set by the slowest stage (stages 1-3 overlap with 4 only if the loader keeps ahead of the GPU).
== probe, GPU 1, args ['--bs', '4', '--train_slices', '8'] (exit 0)
cache /kaggle/input/notebooks/theokorir/knee-v6-k2-cache/cache | 4407 studies | K=2 bs=4 ts=8 workers=2 | 1 torch threads

1 read    : 0.058 s/study | 63 MB/s | 3.7 MB/study | one worker alone would need 3 min/epoch
2 cpu     : 0.176 s/study (convert + augment, 1 process) | 2 workers: 5 min/

In [4]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--torch_threads", "1", "--train_slices", "16", "--bs", "4", "--accum", "2"]
PLAN = {0: [("v6k2_ts16_f0", ["--backbone", CNX_T, "--K", "2", "--bs", "2", "--accum", "4"])],
        1: [("v6k2_ts8_f0",  ["--backbone", CNX_T, "--K", "2", "--train_slices", "8"])]}
NAMES = [n for jobs in PLAN.values() for n, _ in jobs]
FIRST_BEAT_S, STALL_S, EPOCH_BUDGET_S = 15 * 60, 15 * 60, 1300      # no first heartbeat / no progress for 15 min; epoch budget in seconds
LOGS, RC = {}, {}
def lines_of(f): return f.read_text().splitlines() if f is not None and f.exists() else []
def n_epochs(f): return sum("ep " in l for l in lines_of(f))                  # epoch lines: "[f0] ep 3/12 ..."
def n_beats(f): return sum("] it " in l for l in lines_of(f))                 # heartbeats: "[f0] it 25/881 ..."
def last_progress(f):
    ls = [l for l in lines_of(f) if "ep " in l or "] it " in l]
    return ls[-1] if ls else "(not started or no progress line yet)"
def run_one(gpu, name, jx):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); RC[name] = 0; return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + jx + ["--only_folds", "0", "--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    fh = open(LOGS[name], "w")
    p = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE))
    def kill(msg):
        p.kill(); p.wait(); RC[name] = msg; print(f"[{name}] {msg}", flush=True); fh.close()
    t0 = last_t = time.time(); seen_e = seen_total = 0
    while p.poll() is None:
        time.sleep(30)
        ne, nb = n_epochs(LOGS[name]), n_beats(LOGS[name])
        if ne + nb > seen_total:
            seen_total, last_t = ne + nb, time.time()
        if ne > seen_e:
            seen_e = ne
            mm = re.search(r"\| (\d+)s\s*$", [l for l in lines_of(LOGS[name]) if "ep " in l][-1])
            if mm and int(mm.group(1)) > EPOCH_BUDGET_S:
                return kill(f"KILLED: epoch took {mm.group(1)}s (> {EPOCH_BUDGET_S}s budget)")
        beats = [l for l in lines_of(LOGS[name]) if "] it " in l]
        if beats and seen_e == 0:                                               # projected epoch time from the newest heartbeat
            m2 = re.search(r"\] it (\d+)/\d+ .*projected epoch (\d+) min", beats[-1])
            if m2 and int(m2.group(1)) >= 50 and int(m2.group(2)) * 60 > 2 * EPOCH_BUDGET_S:
                return kill(f"KILLED: projected epoch {m2.group(2)} min (> {2 * EPOCH_BUDGET_S // 60} min); last line: {beats[-1]}")
        limit = FIRST_BEAT_S if seen_total == 0 else STALL_S
        if time.time() - (t0 if seen_total == 0 else last_t) > limit:
            return kill("KILLED by watchdog (no progress)")
    fh.close(); RC[name] = p.returncode
    print(f"[{name}] exit code {RC[name]}" + (" (killed by the OS: probably out of RAM)" if RC[name] == -9 else ""), flush=True)
def run_gpu(gpu, jobs):
    for name, jx in jobs: run_one(gpu, name, jx)
threads = [threading.Thread(target=run_gpu, args=(g, jobs)) for g, jobs in PLAN.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(300)
    last = {}
    for n in NAMES:
        last[n] = last_progress(LOGS.get(n))
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished | results:", RC)
for n, rc in RC.items():
    if rc != 0:
        print(f"--- {n} FAILED, log tail ---"); print("\n".join(LOGS[n].read_text().splitlines()[-25:]))

[v6k2_ts16_f0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-v6-k2-cache/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --torch_threads 1 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --K 2 --bs 2 --accum 4 --only_folds 0 --out /kaggle/working/v6k2_ts16_f0
[v6k2_ts8_f0] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-v6-k2-cache/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --torch_threads 1 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --K 2 --train_slices 8 --only_folds 0 --out /kaggle/working/v6k2_ts8_f0
[0.08 h] {
 "v6k2_ts16_f0": "[f0] 

In [5]:
# ---- fold-0 comparison on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
Y = {l: base[f"y_{l}"].values for l in LAB}
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
PRED = {"baseline": base}
for name in NAMES:
    f = W / name / "oof.csv"
    if f.exists():
        d = pd.read_csv(f); assert len(d) == len(base), f"{name}: row count differs from baseline"
        PRED[name.replace("_f0", "")] = d
cx = [p for p in Path("/kaggle/input/notebooks").rglob("oof_v6convnext.csv")]
if cx: PRED["convnext"] = pd.read_csv(cx[0]); print("convnext OOF found:", cx[0])
else: print("convnext OOF not attached: skipping the convnext comparison")
if not any(k not in ("baseline", "convnext") for k in PRED):
    print("No arm produced an oof.csv: both training jobs failed or were killed. Nothing to compare; read the probe output and the log tails above.")
else:
    ref = next(v for k, v in PRED.items() if k not in ("baseline", "convnext"))
    mask = ref[f"p_{LAB[0]}"].notna().values            # fold-0 validation studies = the rows this screen predicted
    print("fold-0 validation studies:", int(mask.sum()))
    RK = {n: {l: rankdata(d.loc[mask, f"p_{l}"].values) / mask.sum() for l in LAB} for n, d in PRED.items()}
    names = [n for n in RK if n not in ("baseline", "convnext")]
    if "convnext" in RK:
        for n in names: RK[f"ens {n}+convnext"] = {l: (RK[n][l] + RK["convnext"][l]) / 2 for l in LAB}
    T = pd.DataFrame({n: {l: auc_l(Y[l][mask], r[l]) for l in LAB} for n, r in RK.items()})
    T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
    print("\nFold 0 only (ranks within fold):\n", T.round(3).to_string())
    print("\nreference from the ConvNeXt run: fold-0 raw macro 0.8328, baseline 0.7795. Seed noise ~0.01.")
    print("rule: K=2 is worth a 5-fold only if an arm beats K=1 ConvNeXt-T fold 0 (0.833 macro / 0.835 KEY-4) by more than ~0.015. Compare the two arms: if ts8 matches ts16, use the cheaper ts8 for the 5-fold. NOTE inference gets slower with K=2 (about twice the windows and 6 series to decode per study).")

convnext OOF found: /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
fold-0 validation studies: 882

Fold 0 only (ranks within fold):
                   baseline  v6k2_ts16  v6k2_ts8  convnext  ens v6k2_ts16+convnext  ens v6k2_ts8+convnext
ACL                  0.833      0.892     0.901     0.892                   0.897                  0.904
MCL                  0.713      0.800     0.786     0.805                   0.807                  0.800
Medial Meniscus      0.757      0.858     0.832     0.870                   0.869                  0.860
Lateral Meniscus     0.715      0.759     0.748     0.776                   0.772                  0.768
Medial OA            0.861      0.881     0.878     0.874                   0.880                  0.880
Lateral OA           0.749      0.782     0.778     0.767                   0.779                  0.779
PF OA                0.722      0.757     0.754     0.740                   0.753                  0.75